In [ ]:
from flask import Flask, render_template, request, jsonify
from services.recommend_service import get_recommendations
from utils.helper import clean_text, format_results, error_message

app = Flask(__name__)

@app.route('/')
def home():
    return render_template('index.html', title="Travel Advisor")

@app.route('/recommend', methods=['POST'])
def recommend():
    try:
        data = request.get_json()
        mood = clean_text(data.get('mood', ''))
        budget = data.get('budget')
        days = data.get('days')

        if not mood:
            return jsonify(error_message("Please enter your mood or interest."))

        results = get_recommendations(mood, budget, days)
        formatted = format_results(results)

        if not formatted:
            return jsonify(error_message("No destinations found."))

        return jsonify({'error': False, 'results': formatted})
    except Exception as e:
        return jsonify(error_message(str(e)))

if __name__ == '__main__':
    app.run(debug=True, use_reloader=False)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [05/Nov/2025 13:36:12] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [05/Nov/2025 13:36:12] "GET /static/css/style.css HTTP/1.1" 200 -
127.0.0.1 - - [05/Nov/2025 13:36:12] "GET /static/js/script.js HTTP/1.1" 200 -
127.0.0.1 - - [05/Nov/2025 13:36:18] "POST /recommend HTTP/1.1" 200 -
127.0.0.1 - - [05/Nov/2025 13:36:18] "GET /static/images/default.jpg HTTP/1.1" 404 -
127.0.0.1 - - [05/Nov/2025 13:36:18] "GET /static/images/default.jpg HTTP/1.1" 404 -
127.0.0.1 - - [05/Nov/2025 13:36:18] "GET /static/images/default.jpg HTTP/1.1" 404 -
127.0.0.1 - - [05/Nov/2025 13:36:18] "GET /static/images/default.jpg HTTP/1.1" 404 -
127.0.0.1 - - [05/Nov/2025 13:36:18] "GET /static/images/default.jpg HTTP/1.1" 404 -
127.0.0.1 - - [05/Nov/2025 13:36:18] "GET /static/images/default.jpg HTTP/1.1" 404 -
127.0.0.1 - - [05/Nov/2025 13:36:18] "GET /static/images/default.jpg HTTP/1.1" 404 -
127.0.0.1 - - [05/Nov/2025 13:36:18] "GET /static/images/de

In [1]:
!python models/mood_model.py


Model training complete — vectorizer and destination data saved successfully!


In [ ]:
r = requests.post("http://127.0.0.1:5000/recommend", json={"mood": "relaxed"})
print(r.json())


In [3]:
import pickle
with open('models/destination_data.pkl', 'rb') as f:
    data = pickle.load(f)

print(data.head())
print(data.columns)



  destination                                        description   category  \
0         Goa  Famous for beaches, nightlife, and Portuguese ...      Beach   
1      Manali  Himalayan town known for snow, adventure sport...  Mountains   
2      Jaipur  Historic city with forts, palaces, and rich Ra...   Heritage   
3    Varanasi  Spiritual city on the Ganges, known for ghats ...  Spiritual   
4      Mumbai  India’s financial capital with diverse culture...      Urban   

              state  avg_cost  \
0               Goa      3500   
1  Himachal Pradesh      3000   
2         Rajasthan      2800   
3     Uttar Pradesh      2000   
4       Maharashtra      4000   

                                            combined  \
0  Famous for beaches, nightlife, and Portuguese ...   
1  Himalayan town known for snow, adventure sport...   
2  Historic city with forts, palaces, and rich Ra...   
3  Spiritual city on the Ganges, known for ghats ...   
4  India’s financial capital with diverse cult

In [ ]:
import requests

url = "http://127.0.0.1:5000/recommend"
payload = {"mood": "relaxed"}

response = requests.post(url, json=payload)
print(response.json())


In [5]:
def get_recommendations(mood):
    mood = mood.lower().strip()
    print(f"Searching for mood: {mood}")
    
    related_terms = mood_map.get(mood, [mood])
    print("Related terms:", related_terms)

    all_results = []
    for term in related_terms:
        query_vec = tfidf.transform([term])
        dest_vecs = tfidf.transform(data['combined'])
        similarity = cosine_similarity(query_vec, dest_vecs).flatten()
        print(f"Top similarity for '{term}':", similarity.max())

        top_idx = similarity.argsort()[-5:][::-1]
        for i in top_idx:
            all_results.append({
                "destination": data.iloc[i]["destination"],
                "category": data.iloc[i]["category"],
                "description": data.iloc[i]["description"],
                "state": data.iloc[i]["state"],
                "avg_cost": data.iloc[i]["avg_cost"]
            })

    print("Total matches:", len(all_results))
    return all_results[:5] if all_results else [{"destination": "No matching destinations found."}]


In [7]:
print(data['combined'].head(10))

0    famous for beaches, nightlife, and portuguese ...
1    himalayan town known for snow, adventure sport...
2    historic city with forts, palaces, and rich ra...
3    spiritual city on the ganges, known for ghats ...
4    india’s financial capital with diverse culture...
5    yoga capital with serene ganges views and adve...
6    french colonial town with beaches and tranquil...
7    high-altitude desert known for monasteries and...
8    tea gardens and scenic himalayan views. mounta...
9    port city with backwaters, heritage sites, and...
Name: combined, dtype: object


In [5]:
import pickle
import pandas as pd

# Load your saved model data (this was created by mood_model.py)
data = pickle.load(open('models/destination_data.pkl', 'rb'))

# Show what columns exist
print(data.columns)

# Show first few rows
print(data.head(10))

# Check if 'combined' column exists
if 'combined' in data.columns:
    print("\n Combined column sample:\n", data['combined'].head(10))
else:
    print("\n 'combined' column missing — need to rerun mood_model.py correctly.")


Index(['destination', 'description', 'category', 'state', 'avg_cost',
       'combined'],
      dtype='object')
   destination                                        description   category  \
0          Goa  Famous for beaches, nightlife, and Portuguese ...      Beach   
1       Manali  Himalayan town known for snow, adventure sport...  Mountains   
2       Jaipur  Historic city with forts, palaces, and rich Ra...   Heritage   
3     Varanasi  Spiritual city on the Ganges, known for ghats ...  Spiritual   
4       Mumbai  India’s financial capital with diverse culture...      Urban   
5    Rishikesh  Yoga capital with serene Ganges views and adve...  Spiritual   
6  Pondicherry  French colonial town with beaches and tranquil...      Beach   
7          Leh  High-altitude desert known for monasteries and...  Mountains   
8   Darjeeling            Tea gardens and scenic Himalayan views.  Mountains   
9        Kochi  Port city with backwaters, heritage sites, and...      Urban   

       

In [1]:
print(final_results[:3])


NameError: name 'final_results' is not defined